## Download Dataset

The dataset is quite large. Make sure to download it on a partition which has large enough space. For this, you can set export HF_DATASETS_CACHE="/your/custom/path" in your .bashrc file. Alternatively, you can train your model with a small shard of the dataset. I have done the latter, and here is the code I used. You may need to lower num_proc depending on how many CPU cores you have on your workstation.

In [ ]:
from datasets import load_dataset
train_dataset = load_dataset("speechbrain/LargeScaleASR", data_files=["small/train-0000*","small/train-0001*"], num_proc=12)
test_dataset = load_dataset("speechbrain/LargeScaleASR", data_files=["test/test-00000*"], num_proc=12)
train_dataset = train_dataset["train"]
test_dataset = test_dataset["train"]
test_dataset = test_dataset.select(range(100)) # only 100 samples used for accelerated testing
print(len(train_dataset))
print(len(test_dataset))

## Clean up

Before we proceed with training the model in the next section, let's clear the current variables and clean the GPU to free up resources.

In [ ]:
import gc
import time
import torch
def clear_memory():
    # Delete variables if they exist in the current global scope
    if 'inputs' in globals(): del globals()['inputs']
    if 'model' in globals(): del globals()['model']
    if 'processor' in globals(): del globals()['processor']
    if 'trainer' in globals(): del globals()['trainer']
    if 'peft_model' in globals(): del globals()['peft_model']
    if 'bnb_config' in globals(): del globals()['bnb_config']
    time.sleep(2)

    # Garbage collection and clearing CUDA memory
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

clear_memory()

# Model finetuning: Stage 1





Since the audio adapters are not trained yet, it makes sense to first freeze the language model and speech encoder and then train only the adapters.

In [ ]:
from transformers import AutoConfig, AutoModelForCausalLM, AutoProcessor
processor = AutoProcessor.from_pretrained(
    "mdmy/whisper_asr_finetuning",
    subfolder="qwen_w_audio_processor",
    trust_remote_code=True
)
BASE_ID = "Qwen/Qwen2-VL-7B-Instruct"
cfg = Qwen2VLConfig.from_pretrained(BASE_ID, trust_remote_code=True)
cfg.use_audio = True
audio_cfg = Qwen2VLAudioConfig()
cfg.audio_config = audio_cfg
model = Qwen2VLForConditionalGenerationWithAudio.from_pretrained(
    "mdmy/whisper_asr_finetuning",
    subfolder="qwen2vl_with_audio_asr",
    trust_remote_code=True
)
model = Qwen2VLForConditionalGenerationWithAudio.from_pretrained(
    "mdmy/whisper_asr_finetuning",
    config = cfg,
    subfolder="qwen2vl_with_audio_asr",
    trust_remote_code=True
)
model.__dict__
# freeze all parameters
for param in model.parameters():
    param.requires_grad = False
allow_patterns = [
        r"^audio_module\.audio_projection(\.|$)",  
        r"\.adapter(\.|$)",                        
        r"\.adapters(\.|$)",
        r"\blora(_|\.|$)",                         
        r"\.lora_[AB](\.|$)",
    ]
allow_res = [re.compile(pattern) for pattern in allow_patterns]
allow_res
def is_allowed(name: str) -> bool:
    return any(r.search(name) for r in allow_res)
trainable = []
for name, param in model.named_parameters():
    if is_allowed(name):
        print(name)
        param.requires_grad = True
        trainable.append(name)
tot = sum(p.numel() for p in model.parameters())
trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trn:,} / {tot:,} ({100.0*trn/max(tot,1):.4f}%)")

In [ ]:
from trl import SFTConfig
# TASK: create an SFT config
SFT COnfig


In [ ]:
import wandb
# TASK: set up wandb.init
connect to wandb

In [ ]:
# TASK: Create a data collator to encode text and audio pairs


In [ ]:
# TASK: Create the SFT trainer and launch training



With the dataset above, I got the following train and eval loss.

![xxx](https://www.dropbox.com/scl/fi/ryjhfsrmn6topw8yqo8n4/phase1.png?rlkey=y80ovdst9nx3xuiniptmwv3vj&st=qfy05frr&raw=1)

In [ ]:
# TASk: push the finetuned model to your HF repo. Note how only 1 out of 4 sharded tensors has changed, why?


## Model finetuning: Stage 2

Now that we have trained the audio projection/adapter, we can finetune the entire model e2e with QLoRA.

QLoRA enables efficient fine-tuning of large language models while significantly reducing the memory footprint compared to traditional methods. Unlike standard LoRA, which reduces memory usage by applying a low-rank approximation, QLoRA takes it a step further by quantizing the model weights. This leads to even lower memory requirements and improved training efficiency, making it an excellent choice for optimizing our model's performance without sacrificing quality.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import BitsAndBytesConfig
from trl import SFTTrainer
# Task: create LoRA, BitsAndBytes, and SFT configs and apply LoRA to the model



In [ ]:
# BitsAndBytesConfig int-4 config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # do the computations in bfloat16
)

# Load model and tokenizer
model_id = "Qwen/Qwen2-VL-7B-Instruct"
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    quantization_config=bnb_config
)

In [ ]:
# Configure LoRA
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=8, # rank needs to be half the alpha
    bias="none",
    target_modules = ["q_proj", "v_proj"], # meaning only language model will be tuned, vision encoder is frozen
    task_type="CAUSAL_LM", # for causal language models aiming at only the text portion
)

In [ ]:
trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_dataset_formatted,
    eval_dataset=eval_dataset_formatted,
    data_collator=collate_fn,
    peft_config=peft_config,
)

In [ ]:
# TASK: set up wand.init and create

In [ ]:
# TASK: Create the SFT trainer and launch training


With the dataset above, I got the following train and eval loss.

![xxx](https://www.dropbox.com/scl/fi/1whmbozm4pcog3ccw0y4g/phase2.png?rlkey=c0ui6y1ng48bjdgsp94p6wme2&st=e5ldl8lt&raw=1)

Let's save and push the results 💾

In [ ]:
# TASk: save and push the finetuned model to your HF repo


# 5. Testing the Fine-Tuned Model 🔍

Now that we've successfully fine-tuned our Audio-Language model, it's time to evaluate its performance! In this section, test the model using your own speech examples.

Recording and Preparing the Audio File
- Record an audio file: Use Voice Memos on Mac to record a speech example.
- Convert to WAV: Use  to convert the file to .wav.
- Resample the audio: Use torchaudio.transforms.Resample to resample the audio to 16K Hz, matching the model's training sampling rate.

Finally save it to an in-memory BytesIO object and include it in tour conversation template and feed to the model for transcription.


Let's clean up the GPU memory to ensure optimal performance 🧹

In [ ]:
clear_memory()

We will reload the base model using the same pipeline as before, but this we will load the LoRA adpaters into the model too.

In [ ]:
# TASK:  Load model, processor, and adapter weights


In [ ]:
import torchaudio
import torchaudio.transforms as T
import io
# TASK: use packages imported above to prepare your .wav file for ASR and run inference



Test the fine-tuned model on the example above, where the model previously struggled to accurately locate the nutrition table.